In [1]:
# pip install urllib3

Note: you may need to restart the kernel to use updated packages.


In [2]:
# pip install websocket

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ------------ --------------------------- 0.5/1.7 MB 2.7 MB/s eta 0:00:01
   ---------------------------------------- 1.7/1.7 MB 5.7 MB/s  0:00:00
  Created wheel for websocket: filename=websocket-0.2.1-py3-none-any.whl size=192185 sha256=4311c4c4dad1fab079d6de694e1e5bb9da2d7c6d0633c25c0c20b92f0d4ed58e
  Stored in directory: c:\users\lenovo\appdata\local\pip\cache\wheels\6f\a5\e6\abc13d6b0eb93225e10a2262a592fe06f308b157d26e11b511
Successfully built websocket

   ---------------- ----------------------- 2/5 [greenlet]
   ------------------------ --------------- 3/5 [gevent]
   ----------------

In [3]:
# pip install websocket_client

Note: you may need to restart the kernel to use updated packages.


In [4]:
# pip install prettytable

Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import sqlite3
import zipfile
import warnings
warnings.filterwarnings('ignore')

In [2]:
import sys
sys.path.append(r"D:/Program Files")   # 一定注意这里是上一级目录

from csmarapi.CsmarService import CsmarService

print("CSMAR 导入成功！")

CSMAR 导入成功！


In [8]:
## 登录CSMAR
from csmarapi.CsmarService import CsmarService
csmar = CsmarService()
csmar.login('hu_liqin@whu.edu.cn','huliqin79626',0)
#csmar.login(account,pwd,lang)
#account：用户名
#pwd：密码
#lang：可选参数（0或1：0代表中文，1代表英文，默认为0）
##显示登陆成功

Thu 04 Dec 2025 16:18:38 CsmarService.py INFO User login succeed


### 1.1 查看数据库基本信息

In [9]:
#如果需要以表格形式展示数据，则需添加
from csmarapi.ReportUtil import ReportUtil
#查看已购买的数据库
database = csmar.getListDbs()
ReportUtil(database)

+--------------------------------------+-----------+---------+
|             databaseName             | startTime | endTime |
+--------------------------------------+-----------+---------+
|   CSMAR 中国上市公司B股财务数据库    |           |         |
|   CSMAR 中国上市公司财务中报数据库   |           |         |
|   CSMAR 中国上市公司财务季报数据库   |           |         |
|   CSMAR 中国上市公司财务年报数据库   |           |         |
| CSMAR 中国上市公司金融行业财务数据库 |           |         |
|           DGTW股票特征基准           |           |         |
|               EVA专题                |           |         |
|           Fama-French因子            |           |         |
|               QFII持股               |           |         |
|               一带一路               |           |         |
|               一致预测               |           |         |
|         上市公司与子公司专利         |           |         |
|           上市公司人物特征           |           |         |
|           上市公司基本信息           |           |         |
|           上市公司研发创新           |         

In [10]:
#查看数据库中数据表名称：
tables = csmar.getListTables('股票市场交易')
ReportUtil(tables)

+-------------------------+------------------------------+------------+------------+
|          table          |          tableName           | startTime  |  endTime   |
+-------------------------+------------------------------+------------+------------+
|          TRD_Co         |           公司文件           | 1990-12-10 | 2025-12-03 |
|         TRD_Cptl        |           分配文件           | 1990-01-01 | 2025-12-12 |
|        TRD_Capchg       |         股本变动文件         | 1990-12-10 | 2025-12-10 |
|        TRD_Dalyr        |       日个股回报率文件       | 1990-12-19 | 2025-12-03 |
|         TRD_Week        |       周个股回报率文件       | 1990-12-17 | 2025-11-26 |
|         TRD_Mnth        |       月个股回报率文件       | 1990-12-01 | 2025-11-01 |
|         TRD_Year        |       年个股回报率文件       | 1990-01-01 | 2024-12-31 |
|         TRD_Cale        |           日历文件           | 1990-12-19 | 2025-12-03 |
|        TRD_Dalym        |       日市场回报率文件       | 1990-12-19 | 2025-12-03 |
|        TRD_Weekm        |       周市场回

In [11]:
#各变量名称
fields = csmar.getListFields('TRD_Co')
ReportUtil(fields)

+-------------------+----------------------------+-----------+----------+----------+
|       field       |         fieldName          | fieldType | ableNull | fieldKey |
+-------------------+----------------------------+-----------+----------+----------+
|      Cuntrycd     |          国家代码          |  decimal  |    NO    |          |
|       Stkcd       |          证券代码          |  varchar  |    NO    |   Code   |
|       Stknme      |          证券简称          |  varchar  |   YES    |          |
|       Conme       |          公司全称          |  varchar  |   YES    |          |
|      Conme_en     |        公司英文全称        |  varchar  |   YES    |          |
|       Indcd       |         行业代码A          |  varchar  |   YES    |          |
|       Indnme      |         行业名称A          |  varchar  |   YES    |          |
|       Nindcd      |         行业代码B          |  varchar  |   YES    |          |
|      Nindnme      |         行业名称B          |  varchar  |   YES    |          |
|      Nnindcd    

### 1.2 查询与下载数据

（1）问题1：一次最多查询20万条数据  

（2）问题2：日度数据只能下载5年  

（3）循环解压

In [13]:
#查询表数据,query_df(columns,condition,tableName,startTime,endTime)
# 示例，查询2010-01-01到2023-12-31期间，股票代码以6开头的公司基本信息表中的国家代码，证券代码，证券简称，公司全称四个变量
data = csmar.query_df(['Cuntrycd','Stkcd','Stknme','Conme'], "Stkcd like'6%'", 'TRD_Co','2010-01-01','2023-12-31')
#data = csmar.query_df(['Cuntrycd','Stkcd','Stknme','Conme'],'','TRD_Co','2010-01-01','2022-12-31')###无condition
data

,Cuntrycd,Stkcd,Stknme,Conme
0,10,600023,浙能电力,浙江浙能电力股份有限公司
1,10,600025,华能水电,华能澜沧江水电股份有限公司
2,10,600032,浙江新能,浙江省新能源投资集团股份有限公司
3,10,600901,江苏金租,江苏金融租赁股份有限公司
4,10,600903,贵州燃气,贵州燃气集团股份有限公司
...,...,...,...,...
1467,10,688799,华纳药厂,湖南华纳大药厂股份有限公司
1468,10,688800,瑞可达,苏州瑞可达连接系统股份有限公司
1469,10,688819,天能股份,天能电池集团股份有限公司
1470,10,688981,中芯国际,中芯国际集成电路制造有限公司


In [18]:
# 示例，查询2020-01-01到2025-11-30期间，股票代码以6开头的日交易数据表中的证券代码，交易日期，收盘价三个变量，从第一条开始读取读取1000条记录
data = csmar.query_df(['Stkcd','Trddt','Clsprc'], "Stkcd like'6%' limit 0,1000" , 'TRD_Dalyr','2020-01-01','2025-11-30')
data

Thu 04 Dec 2025 16:34:14 CsmarService.py ERROR The date range for selecting this table should be within 5 years


In [15]:
#查看数据行数
# 示例：查询2020-12-01到2022-12-31期间，股票代码以3开头的日交易数据表中的证券代码，交易日期，收盘价三个变量的记录数（需要时间较长）
csmar.queryCount(['Stkcd','Trddt','Clsprc'], "Stkcd like'3%'", 'TRD_Dalyr' ,'2020-12-01','2022-12-31')

Thu 04 Dec 2025 16:23:35 CsmarService.py INFO The total number of data obtained by condition is 539140


539140

In [16]:
from datetime import datetime, timedelta

def download_csmar_by_years(fields, condition, table, start_date_str, end_date_str, interval_years=4):
    """
    按“若干年一个区间”分段下载 CSMAR 数据的通用函数。
    
    参数说明：
    - fields          : list，要下载的字段名列表，例如 ['Stkcd', 'Trddt', 'Clsprc']
    - condition       : str，筛选条件（SQL 风格），例如 "Stkcd like '3%'"
    - table           : str，表名，例如 'TRD_Dalyr'
    - start_date_str  : str，起始日期字符串，格式 'YYYY-MM-DD'
    - end_date_str    : str，结束日期字符串，格式 'YYYY-MM-DD'
    - interval_years  : int，每个区间跨多少年，例如 4 表示“每 4 年一个区间”
    
    说明：
    - 本函数会把 [start_date, end_date] 之间的时间拆成若干个区间，
      每个区间大致是 interval_years 年，然后对每个区间调用一次
      csmar.getPackResultExt()。
    - 下载得到的压缩包默认保存在 C:\\csmardata\\zip 目录下（由 CSMAR 客户端管理）。
    """
    
    # 1. 将字符串形式的起止日期转换为 datetime 对象，便于做日期运算
    start_date = datetime.strptime(start_date_str, "%Y-%m-%d")
    end_date = datetime.strptime(end_date_str, "%Y-%m-%d")
    
    # 2. 当前区间的开始日期，一开始设为整体起始日
    current_start_date = start_date

    # 3. 只要当前区间的开始日期还早于整体结束日期，就持续循环
    while current_start_date < end_date:
        # 3.1 计算当前区间的“理论结束日期”
        #     思路：以当前开始日期所在年份 + interval_years，月份固定为 12，日期固定为 31
        #     例如：当前起始年为 2010，interval_years = 4，则初始结束日为 2014-12-31
        current_end_date = datetime(current_start_date.year + interval_years, 12, 31)

        # 3.2 如果理论结束日期超过了整体 end_date，则用整体 end_date 截断
        #     这样可以避免最后一个区间跑到数据范围之外
        if current_end_date > end_date:
            current_end_date = end_date

        # 3.3 将 datetime 对象格式化为 'YYYY-MM-DD' 字符串，
        #     因为 CSMAR 接口需要的是字符串形式的日期
        start = current_start_date.strftime("%Y-%m-%d")
        end = current_end_date.strftime("%Y-%m-%d")

        # 3.4 打印当前区间的起止日期，方便在控制台观察进度和检查是否分段正确
        print("开始日期:", start)
        print("结束日期:", end)

        # 3.5 调用 CSMAR 接口，按当前区间进行一次下载
        #     - fields    : 要下载的列名列表
        #     - condition : 筛选条件（如股票代码以 3 开头）
        #     - table     : CSMAR 表名
        #     - start/end : 当前区间的起止日期
        csmar.getPackResultExt(
            fields,
            condition,
            table,
            start,
            end
        )

        # 3.6 提示本次区间下载已完成
        print("本区间下载完成\n")

        # 3.7 更新 current_start_date 为“下一段区间”的开始日期
        #     方式：在当前区间结束日基础上 +1 天
        #     例如：当前是 2014-12-31，则下一段从 2015-01-01 开始
        current_start_date = current_end_date + timedelta(days=1)

    # 4. 循环结束后，说明所有区间都已下载完毕
    print("所有区间下载完成！")

In [17]:
# ==========================
# 示例：调用函数
# ==========================
# 假定你已经在前面完成 CSMAR 登录：
# from csmarapi.CsmarService import CsmarService
# csmar = CsmarService()
# csmar.login('你的账号', '你的密码', 0)
# 例子：从 2020-01-01 到 2025-11-30，
#      每 4 年为一个区间，下载 TRD_Dalyr 表中
#      股票代码以 3 开头的股票的 代码/日期/收盘价 三个字段
download_csmar_by_years(
    fields=['Stkcd', 'Trddt', 'Clsprc'],   # 要下载的字段列表
    condition="Stkcd like '3%'",           # 筛选条件：股票代码以 3 开头
    table='TRD_Dalyr',                     # 表名：日度股票交易数据
    start_date_str="2020-01-01",           # 整体起始日期
    end_date_str="2025-11-30",             # 整体结束日期
    interval_years=4                       # 每 4 年一个区间
)

开始日期: 2020-01-01
结束日期: 2024-12-31


Thu 04 Dec 2025 16:31:59 CsmarService.py INFO packaging return code：signCode=1446177223195267072
Thu 04 Dec 2025 16:31:59 CsmarService.py INFO downloading...


|▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇|100%

Thu 04 Dec 2025 16:33:31 CsmarService.py INFO Package successfully. File size is：6.9MB


Thu 04 Dec 2025 16:33:56 CsmarService.py INFO Generate the local file path as ：c:\csmardata\zip\1446177223195267072.zip
Thu 04 Dec 2025 16:33:56 CsmarService.py INFO packaging return code：signCode=1446177712037203968


...

Thu 04 Dec 2025 16:33:56 CsmarService.py INFO downloading...


|▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇|100%

Thu 04 Dec 2025 16:34:09 CsmarService.py INFO Package successfully. File size is：1.5MB


Thu 04 Dec 2025 16:34:14 CsmarService.py INFO Generate the local file path as ：c:\csmardata\zip\1446177712037203968.zip


本区间下载完成

所有区间下载完成！


In [19]:
import os

# getPackResultExt() 是 CSMAR 客户端封装好的 API
# • 它内部通过客户端服务进行数据请求
# • 下载后的文件由客户端放进 C:\csmardata\zip
zip_path = r"C:\csmardata\zip"  

files = [f for f in os.listdir(zip_path) if f.endswith(".zip")]

print("当前下载的文件：")
for f in files:
    print(f)

当前下载的文件：
1446177223195267072.zip
1446177712037203968.zip


In [21]:
import os

# 建议把数据库放在一个安全的英文路径
db_dir = r"D:\csmar_sqlite"
os.makedirs(db_dir, exist_ok=True)

print("数据库文件夹已确认存在：", db_dir)

数据库文件夹已确认存在： D:\csmar_sqlite


In [22]:
# 链接数据库 

import sqlite3
import os

db_path = r"D:\csmar_sqlite\csmar_data.sqlite"

# 创建并连接数据库
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

print("数据库连接成功！文件位置：", db_path)

数据库连接成功！文件位置： D:\csmar_sqlite\csmar_data.sqlite


In [23]:
# 自动解压所有zip
import os, zipfile

zip_dir = r"C:\csmardata\zip"
out_dir = r"C:\csmardata\unzip"
os.makedirs(out_dir, exist_ok=True)

for f in os.listdir(zip_dir):
    if f.endswith(".zip"):
        zip_path = os.path.join(zip_dir, f)
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(out_dir)
        print(f"{f} 解压完成")

1446177223195267072.zip 解压完成
1446177712037203968.zip 解压完成


In [24]:
import pandas as pd

# 1. 指定 CSV 文件路径（修改成你自己的文件名）
csv_file = r"C:\csmardata\unzip\TRD_Dalyr.csv"

# 2. 使用 pandas 加载 CSV 文件
df = pd.read_csv(csv_file, encoding='utf-8')   # 若报错换成 encoding='gbk'

print("CSV 加载成功，行数：", len(df))

# 3. 写入数据库（表名可自定义）
df.to_sql(
    name="TRD_Dalyr",    # 表名（你可以改）
    con=conn,            # 数据库连接
    if_exists="replace", # 若表已存在，替换
    index=False          # 不写入行索引
)

print("数据已成功写入 SQLite 数据库！")

CSV 加载成功，行数： 302872
数据已成功写入 SQLite 数据库！


In [25]:
# 查看数据库里有哪些表
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()

print("数据库中的表：")
for t in tables:
    print("-", t[0])

数据库中的表：
- TRD_Dalyr


In [26]:
# 查看 TRD_Dalyr 表的前几行（像 pandas.head）
import pandas as pd

df_head = pd.read_sql("SELECT * FROM TRD_Dalyr LIMIT 10;", conn)
df_head

,Stkcd,Trddt,Clsprc
0,300001,2025-01-02,20.94
1,300001,2025-01-03,20.31
2,300001,2025-01-06,20.40
3,300001,2025-01-07,20.80
4,300001,2025-01-08,20.74
5,300001,2025-01-09,20.92
6,300001,2025-01-10,20.80
7,300001,2025-01-13,20.59
8,300001,2025-01-14,21.66
9,300001,2025-01-15,22.35


In [20]:
# 链接数据库
camar_sql = sqlite3.connect(database='data/camar_python.sqlite')#此时data文件夹下会有一个sqlite文件

OperationalError: unable to open database file